In [8]:
%use kandy
import profilelib.*

We are trying to estimate the performance ratio of stdlib functions in Kotlin/Native vs. JVM. For this example, we focus
on `String.substring` method used in this particular scenario. We need to exclude the effect of GC to get the performace
ratio of execution time of the function itself. The following cell describes, how we collect the data. We
end up with a list of execution times.

In [9]:
import java.nio.file.Files
import java.nio.file.Path
import java.nio.file.StandardOpenOption

val kotlincNative = System.getProperty("user.home") + "/.konan/kotlin-native-prebuilt-linux-x86_64-2.2.10/bin/kotlinc-native"
val tempDir = Files.createTempDirectory("knative-perf")

fun runProcess(vararg command: String): String =
    ProcessBuilder(command.toList())
        .directory(tempDir.toFile())
        .start()
        .let { process ->
            process.inputStream.bufferedReader().readText()
                .also {
                    if (process.waitFor() != 0) {
                        process.errorStream.bufferedReader().readText().let(::println)
                        throw Exception("Process failed with code ${process.exitValue()}")
                    }
                }
        }

val source = """
import kotlin.concurrent.Volatile
import kotlin.time.measureTime
import kotlin.time.DurationUnit
import kotlin.time.Duration
import kotlin.native.runtime.GC

@Volatile
private var blackhole: Any? = null
@Volatile
private var string = "abc".repeat(100000)

@kotlin.native.runtime.NativeRuntimeApi
@kotlin.ExperimentalStdlibApi
fun main() {
    var epoch: Long? = null
    repeat(1_000_000) { blackhole = string.substring(1000, 5000) }
    val iterations = 10_000
    val times = ArrayList<Duration?>(iterations)
    repeat(iterations) {
        measureTime {
            repeat(100) {
                blackhole = string.substring(1000, 5000)
            }
        }.let { times.add(it) }
        epoch = GC.lastGCInfo?.epoch.also {
            if (it != epoch) times.add(null)
        }
    }
    times.forEach { println(it?.toDouble(DurationUnit.SECONDS)) }
}
"""

val sourceFile = tempDir.resolve("benchmark.kt")
val outputBinary = tempDir.resolve("benchmark.kexe")

Files.writeString(
    sourceFile,
    source,
    StandardOpenOption.CREATE,
)

runProcess(
        kotlincNative,
        sourceFile.toString(),
        "-o", outputBinary.toString(),
)

val times = runProcess(outputBinary.toString())
    .split("\n")
    .mapNotNull { it.takeIf { it.isNotEmpty() } }
    .map { it.toDoubleOrNull() }
    .zipWithNext()
    .partition { it.second == null }
    .let { (x, y) -> mapOf(
        "GC" to x.map { it.first!! },
        "noGC" to y.mapNotNull { it.first },
        )}


[2.1833E-5, null, 2.5295E-5, 2.2045E-5, 2.1117E-5, 2.3226E-5, 1.07427E-4, null, 2.0878E-5, 2.3126E-5, 2.2009E-5, 2.1752E-5, 1.33007E-4, null, 2.712E-5, 2.0343E-5, 2.3615E-5, 2.0735E-5, 2.6626E-5, 9.909E-5, null, 2.2163E-5, 2.1039E-5, 2.2478E-5, 2.1145E-5, 8.3346E-5, 2.4634E-5, null, 2.2998E-5, 2.058E-5, 2.1447E-5, 4.2087E-5, 7.1429E-5, null, 1.9567E-5, 1.9277E-5, 2.3771E-5, 2.2346E-5, 9.8761E-5, null, 2.2996E-5, 2.1654E-5, 2.2632E-5, 2.1116E-5, 1.21264E-4, null, 2.4597E-5, 2.2234E-5, 2.2229E-5, 2.1787E-5, 3.9879E-5, 7.2768E-5, null, 2.0217E-5, 2.0239E-5, 2.2696E-5, 9.1509E-5, 2.8116E-5, null, 2.5807E-5, 2.2791E-5, 2.122E-5, 2.2265E-5, 7.7898E-5, null, 2.6058E-5, 2.2248E-5, 2.0811E-5, 2.1124E-5, 7.8259E-5, 2.8238E-5, null, 2.0371E-5, 2.2334E-5, 2.1068E-5, 8.119E-5, null, 2.8701E-5, 2.0212E-5, 2.0747E-5, 2.4198E-5, 7.5589E-5, 6.9559E-5, null, 1.9865E-5, 2.3713E-5, 2.1067E-5, 2.222E-5, 9.8039E-5, null, 2.5427E-5, 2.2079E-5, 2.106E-5, 2.1514E-5, 8.0953E-5, null, 2.6613E-5, 2.3358E-5, 2.181

For the exclusion of GC, we assume that the execution time distribution is bimodal (i .e. there are two peaks). The
first peak is formed by the executions that didn't trigger GC. The second peak is formed by the executions that
triggered GC. By finding a threshold for the execution time that lies between the two peaks, we can separate the
executions into those that ran GC and those which didn't.

In [10]:
plot_histogram(times)

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="CmeEgV"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("CmeEgV");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"name":"sample",
"limits":[null,null]
},{
"aesthetic":"x",
"limits":[null,null]
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"fill",
"discrete":true
}],
"layers":[{
"mapping":{
"x":"x",
"y":"count",
"fill":"group",
"group":"&merged_groups"
},
"stat":"identity",
"data":{
"&merged_groups":["GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC"],
"x":[1.460225E-5,4.3806750000000005E-5,7.301125E-5,1.0221575000000001E-4,1.3142025000000002E-4,1.6062475E-4,1.8982925E-4,2.1903375000000003E-4,2.4823825E-4,2.7744275E-4,3.0664725000000004E-4,3.3585175E-4,3.6505625E-4,3.9426075000000005E-4,4.2346525E-4,4.5266975E-4,4.8187425E-4,5.1107875E-4,5.4028325E-4,5.694877500000001E-4,1.460225E-5,4.3806750000000005E-5,7.301125E-5,1.0221575000000001E-4,1.3142025000000002E-4,1.6062475E-4,1.8982925E-4,2.1903375000000003E-4,2.4823825E-4,2.7744275E-4,3.0664725000000004E-4,3.3585175E-4,3.6505625E-4,3.9426075000000005E-4,4.2346525E-4,4.5266975E-4,4.8187425E-4,5.1107875E-4,5.4028325E-4,5.694877500000001E-4],
"count":[189.0,100.0,575.0,321.0,318.0,180.0,153.0,16.0,8.0,6.0,0.0,3.0,4.0,3.0,2.0,2.0,2.0,7.0,2.0,4.0,7068.0,329.0,363.0,228.0,103.0,10.0,2.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0],
"group":["GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","GC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC","noGC"]
},
"sampling":"none",
"inherit_aes":false,
"position":"dodge",
"geom":"bar",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"float",
"column":"x"
},{
"type":"int",
"column":"count"
},{
"type":"str",
"column":"&merged_groups"
}]
}
}],
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"group"
},{
"type":"str",
"column":"&merged_groups"
}]
},
"spec_id":"2"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 


We then estimate the total time spent in GC by the following formula.

In [11]:
val avgNoGC = times["noGC"]!!.average()
times["GC"]!!.sumOf { it - avgNoGC } / (times["noGC"]!!.sum() + times["GC"]!!.sum())

0.3364651930549282

For measurements that run at least 100 000 iterations and don't print the times throught the execution (but all at once at the end), we estimate that GC is responsible for over 80% of the execution. That is insane. Therefore, we question the results.

* Can GC in K/N be responsible for such a high percentage of execution time? After all, this is very memory heavy benchmark.
* Is there an error in the benchmark setup, collection of the data or the processing?
* Is any one of our assumptions incorrect?
* Is this method for estimating GC time sound?